In [56]:
from fetch_data import load_all

data = load_all()
income_records = data["hh_income_state"]
poverty_records = data["hh_poverty_state"]

print(f"Income records: {len(income_records)}")
print(f"Poverty records: {len(poverty_records)}")

Income records: 319
Poverty records: 310


In [57]:
"""Shared functions for Assignment 1."""

import numpy as np

SURVEY_YEARS = [2016, 2019, 2020, 2022, 2024]

POVERTY_YEARS = [2019, 2020, 2022, 2024]

STATES = {
    "Northern": ["Perlis", "Kedah", "Pulau Pinang", "Perak"],
    "Central": [
        "Selangor",
        "W.P. Kuala Lumpur",
        "W.P. Putrajaya",
        "Negeri Sembilan",
    ],
    "Southern": ["Melaka", "Johor"],
    "East Coast": ["Kelantan", "Terengganu", "Pahang"],
    "East Malaysia": ["Sabah", "Sarawak", "W.P. Labuan"],
}


def get_year(record: dict) -> int:
    """Get survey year from a record.
    Owner: Role 2. Reviewed by: Role 1.
    """
    return int(record["date"][:4])


def to_float(value) -> float:
    """Convert None to np.nan and other values to float.
    Owner: Role 2. Reviewed by: Role 1.
    """
    if value is None:
        return np.nan
    return float(value)

In [58]:
STATES = sorted(set(record["state"] for record in income_records))
print(STATES)
print(len(STATES))

['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']
16


In [59]:
matric_numbers=[314171,313819,313374,313994,313989]
names= ["Naurah", "Irdina", "Humaira", "Ainul", "Khairina"]
used_states=set()
GROUP_STATES=[None]*len(matric_numbers)

for i in sorted(range(len(matric_numbers)) ,
                key=lambda i: matric_numbers[i]):

  start = matric_numbers[i] % 16

  for offset in range(16): 
    state = STATES[(start + offset) % 16]
    if state not in used_states: 
        GROUP_STATES[i] = state 
        used_states.add(state)
        break
    
for name, matric, state in zip(names, matric_numbers, GROUP_STATES):
   print(name, matric, "->", state)

Naurah 314171 -> Terengganu
Irdina 313819 -> Selangor
Humaira 313374 -> W.P. Labuan
Ainul 313994 -> Sarawak
Khairina 313989 -> Pahang


In [60]:
import numpy as np

states = sorted({
    r["state"]
    for r in income_records
    if get_year(r) in SURVEY_YEARS
})

income_matrix = np.array([
    [
        to_float(
            next(r["income_median"]
                 for r in income_records
                 if r["state"] == state and get_year(r) == year)
        )
        for year in SURVEY_YEARS
    ]
    for state in states
])

print(income_matrix.shape)

(16, 5)


In [61]:
print("TASK 4.1")

year_means = income_matrix.mean(axis=0)
year_medians = np.median(income_matrix, axis=0)

for year, mean, median in zip(
        SURVEY_YEARS,
        year_means,
        year_medians):

    print(f"{year}:")
    print(f"Mean = {mean:.2f}")
    print(f"Median = {median:.2f}\n")


TASK 4.1
2016:
Mean = 5235.94
Median = 4636.50

2019:
Mean = 5915.12
Median = 5275.00

2020:
Mean = 5276.19
Median = 4634.00

2022:
Mean = 6212.69
Median = 5552.00

2024:
Mean = 6741.94
Median = 6109.00



This is not Malaysia's median household income because it is calculated from 16 state medians rather than from all Malaysian households. The median of state medians does not represent the national household distribution directly.

In [62]:
print("TASK 4.2")

growth = (
    (income_matrix[:, 4] - income_matrix[:, 0])
    / income_matrix[:, 0]
    * 100
)

for state, g in zip(states, growth):
    print(f"{state}: {g:.2f}%")

highest = np.argmax(growth)
lowest = np.argmin(growth)

print(
    "Highest growth:",
    states[highest],
    f"({growth[highest]:.2f}%)"
)

print(
    "Lowest growth:",
    states[lowest],
    f"({growth[lowest]:.2f}%)"
)

TASK 4.2
Johor: 36.45%
Kedah: 28.44%
Kelantan: 32.61%
Melaka: 23.32%
Negeri Sembilan: 22.10%
Pahang: 25.03%
Perak: 17.00%
Perlis: 17.75%
Pulau Pinang: 36.55%
Sabah: 18.98%
Sarawak: 32.21%
Selangor: 48.46%
Terengganu: 41.18%
W.P. Kuala Lumpur: 19.06%
W.P. Labuan: 24.54%
W.P. Putrajaya: 30.14%
Highest growth: Selangor (48.46%)
Lowest growth: Perak (17.00%)


In [63]:
print("TASK 4.3")

for i, year in enumerate(SURVEY_YEARS):

    high = np.argmax(income_matrix[:, i])
    low = np.argmin(income_matrix[:, i])

    print(
        f"{year} -> highest: {states[high]}"
        f" / lowest: {states[low]}"
    )

TASK 4.3
2016 -> highest: W.P. Kuala Lumpur / lowest: Kelantan
2019 -> highest: W.P. Kuala Lumpur / lowest: Kelantan
2020 -> highest: W.P. Putrajaya / lowest: Kelantan
2022 -> highest: W.P. Kuala Lumpur / lowest: Kelantan
2024 -> highest: W.P. Kuala Lumpur / lowest: Kelantan


In [64]:
#TASK 4.4

year_medians = np.median(
    income_matrix,
    axis=0
)

income_index = income_matrix / year_medians

print("TASK 4.5")

mask = (
    (income_index[:, 4] < 0.8)
    &
    (income_index[:, 4] < income_index[:, 0])
)

target_states = np.array(states)[mask]
print("States meeting both conditions:")

for state in target_states:
    print(state)

TASK 4.5
States meeting both conditions:
Perak
